# LeetCode 2115 — Find All Possible Recipes from Given Supplies bằng Python

Link đề: https://leetcode.com/problems/find-all-possible-recipes-from-given-supplies/ · Độ khó: **Medium** · Chủ đề: Graph, Topological Sort, Array, Hash Table, String, Directed Acyclic Graph

Skeleton LeetCode:

```python
class Solution:
    def findAllRecipes(self, recipes: List[str], ingredients: List[List[str]], supplies: List[str]) -> List[str]:
        ...
```

## 1. Mục tiêu học

1. Nhận ra một bài "nấu ăn" thực chất là **đồ thị phụ thuộc** (dependency graph): cạnh `nguyên liệu → món cần nó`.
2. Hiểu và cài đặt **thuật toán Kahn** (topological sort bằng BFS với bậc vào — *in-degree*) từ con số 0.
3. Cài đặt lời giải **DFS + ghi nhớ (memo) + 3 trạng thái màu** để phát hiện chu trình, và hiểu vì sao thiếu trạng thái "đang thăm" sẽ gây lặp vô hạn hoặc kết quả sai.
4. So sánh với cách **lặp đến khi ổn định** (fixed-point iteration) — baseline dễ nghĩ nhất — và chỉ ra nút thắt của nó.
5. Thành thạo `dict`, `set`, `defaultdict`, `deque` trong Python để xây đồ thị từ chuỗi.

## 2. Đề bài & ví dụ

Có `n` **món ăn** (recipe). Món thứ `i` tên là `recipes[i]` và cần **toàn bộ** nguyên liệu trong `ingredients[i]` mới nấu được. Một món ăn cũng có thể là **nguyên liệu** của món khác (ví dụ "sandwich" cần "bread", mà "bread" lại là một món).

Mảng `supplies` là các nguyên liệu bạn **có sẵn ngay từ đầu**, với số lượng **vô hạn**.

Hãy trả về **danh sách mọi món có thể nấu được** (thứ tự tùy ý). Lưu ý: hai món có thể cần lẫn nhau (chu trình) — khi đó cả hai đều không nấu được (trừ khi... không có cách nào khác, vì mỗi món chỉ có đúng một công thức).

**Ví dụ 1:** `recipes = ["bread"]`, `ingredients = [["yeast","flour"]]`, `supplies = ["yeast","flour","corn"]` → `["bread"]`.

**Ví dụ 2:** `recipes = ["bread","sandwich"]`, `ingredients = [["yeast","flour"],["bread","meat"]]`, `supplies = ["yeast","flour","meat"]` → `["bread","sandwich"]`. Sandwich cần bread, mà bread nấu được.

**Ví dụ 3:** thêm `"burger"` cần `["sandwich","meat","bread"]` → `["bread","sandwich","burger"]`.

**Ràng buộc:**
- `n == recipes.length == ingredients.length`, `1 ≤ n ≤ 100`
- `1 ≤ ingredients[i].length, supplies.length ≤ 100`
- Mỗi chuỗi dài 1..10, chỉ chữ thường.
- Tất cả giá trị của `recipes` và `supplies` gộp lại là **phân biệt** (một tên không vừa là món vừa là nguyên liệu có sẵn).
- Mỗi `ingredients[i]` không có phần tử trùng.

In [1]:
from typing import List
from collections import defaultdict, deque
import random, time

examples = [
    (["bread"], [["yeast","flour"]], ["yeast","flour","corn"], ["bread"]),
    (["bread","sandwich"], [["yeast","flour"],["bread","meat"]], ["yeast","flour","meat"], ["bread","sandwich"]),
    (["bread","sandwich","burger"], [["yeast","flour"],["bread","meat"],["sandwich","meat","bread"]],
     ["yeast","flour","meat"], ["bread","sandwich","burger"]),
]
for i, (r, ing, sup, exp) in enumerate(examples, 1):
    print(f"Ví dụ {i}: recipes={r}")
    for name, need in zip(r, ing):
        print(f"   {name:<9} cần {need}")
    print(f"   supplies={sup}  -> kỳ vọng {exp}")

Ví dụ 1: recipes=['bread']
   bread     cần ['yeast', 'flour']
   supplies=['yeast', 'flour', 'corn']  -> kỳ vọng ['bread']
Ví dụ 2: recipes=['bread', 'sandwich']
   bread     cần ['yeast', 'flour']
   sandwich  cần ['bread', 'meat']
   supplies=['yeast', 'flour', 'meat']  -> kỳ vọng ['bread', 'sandwich']
Ví dụ 3: recipes=['bread', 'sandwich', 'burger']
   bread     cần ['yeast', 'flour']
   sandwich  cần ['bread', 'meat']
   burger    cần ['sandwich', 'meat', 'bread']
   supplies=['yeast', 'flour', 'meat']  -> kỳ vọng ['bread', 'sandwich', 'burger']


## 3. Kiến thức nền

### 3.1 `set` — tra cứu "đã có chưa" trong O(1)

Ta cần hỏi liên tục "nguyên liệu `x` đã có chưa?". Với `list`, phép `x in lst` tốn O(độ dài). Với `set`, trung bình O(1) nhờ bảng băm.

In [2]:
have = set(["yeast", "flour", "meat"])
print("flour" in have, "bread" in have)
have.add("bread")            # nấu xong bread -> giờ đã có bread
print("bread" in have, len(have))

True False
True 4


### 3.2 `defaultdict(list)` và `defaultdict(int)`

Khi xây đồ thị, ta muốn `g[v].append(...)` mà không phải kiểm tra `v` đã có khóa chưa. `defaultdict(list)` tự tạo `[]` khi truy cập khóa mới; `defaultdict(int)` tự tạo `0`.

**Cẩn thận:** chỉ cần *đọc* `g[v]` cũng đã tạo khóa mới. Nếu muốn đọc mà không tạo, dùng `g.get(v, [])`.

In [3]:
g = defaultdict(list)
g["flour"].append("bread")
g["yeast"].append("bread")
print(dict(g))
_ = g["corn"]                # chỉ đọc thôi cũng tạo khóa "corn": []
print(dict(g))
print(g.get("salt", []), "salt" in g)   # .get không tạo khóa

{'flour': ['bread'], 'yeast': ['bread']}
{'flour': ['bread'], 'yeast': ['bread'], 'corn': []}
[] False


### 3.3 Đồ thị có hướng và **bậc vào** (in-degree)

Đồ thị có hướng gồm các đỉnh và các cạnh một chiều `u → v`. Ở bài này:
- Đỉnh = tên (nguyên liệu hoặc món).
- Cạnh `x → r` nghĩa là "món `r` cần `x`".
- **Bậc vào** của món `r` = số nguyên liệu nó cần = `len(ingredients[i])`.

Một món **nấu được** khi tất cả cạnh đi vào nó đều đến từ những thứ "đã có". Mỗi lần một nguyên liệu trở thành "đã có", ta **giảm bậc vào** của mọi món cần nó đi 1. Khi bậc vào về 0 ⇒ món đó nấu được.

In [4]:
recipes = ["bread", "sandwich"]
ingredients = [["yeast", "flour"], ["bread", "meat"]]
g = defaultdict(list); indeg = {}
for r, need in zip(recipes, ingredients):
    indeg[r] = len(need)
    for x in need:
        g[x].append(r)
print("cạnh  :", {k: v for k, v in g.items()})
print("indeg :", indeg)

cạnh  : {'yeast': ['bread'], 'flour': ['bread'], 'bread': ['sandwich'], 'meat': ['sandwich']}
indeg : {'bread': 2, 'sandwich': 2}


### 3.4 `deque` — hàng đợi hai đầu

BFS cần hàng đợi: lấy ở đầu, thêm ở cuối. `list.pop(0)` tốn O(n) vì phải dời mọi phần tử; `deque.popleft()` tốn O(1).

Mẹo Python: duyệt `for x in lst:` trong khi `lst.append(...)` **vẫn hoạt động** (vòng for đọc theo chỉ số, thấy được phần tử mới thêm). Lời giải doocs dùng đúng mẹo này để khỏi cần `deque`. Tuy nhiên với người mới, `deque` rõ ràng hơn.

In [5]:
q = deque(["yeast", "flour"])
q.append("meat")
print(q.popleft(), list(q))

lst = [1, 2]
seen = []
for x in lst:
    seen.append(x)
    if x < 4:
        lst.append(x + 2)     # phần tử thêm vào vẫn được duyệt tới
print(seen)

yeast ['flour', 'meat']
[1, 2, 3, 4, 5]


### 3.5 Đệ quy + memo + "3 màu"

Khi DFS trên đồ thị có thể có chu trình, ta gán mỗi đỉnh một trạng thái:
- **0 (trắng)**: chưa thăm.
- **1 (xám)**: đang thăm (đang nằm trên ngăn xếp đệ quy).
- **2 (đen)**: đã xong, kết quả đã ghi vào memo.

Nếu đang DFS mà gặp lại một đỉnh **xám** ⇒ có chu trình. Ở bài này, món nằm trong chu trình **không bao giờ** nấu được (vì muốn nấu nó phải có nó trước).

In [6]:
# Ví dụ nhỏ: phát hiện chu trình a -> b -> a
edges = {"a": ["b"], "b": ["a"], "c": []}
state = {}
def has_cycle(u):
    state[u] = 1
    for v in edges[u]:
        if state.get(v, 0) == 1:
            return True
        if state.get(v, 0) == 0 and has_cycle(v):
            return True
    state[u] = 2
    return False
print("a:", has_cycle("a"))
state.clear(); print("c:", has_cycle("c"))

a: True
c: False


## 4. Ý tưởng nền tảng / trực giác

Hãy tưởng tượng bạn đứng trong bếp. Bạn có sẵn `yeast, flour, meat`. Bạn nhìn danh sách công thức (ví dụ 3):

| Món | Cần | Thiếu gì lúc đầu? |
|---|---|---|
| bread | yeast, flour | không thiếu → nấu ngay |
| sandwich | bread, meat | thiếu bread |
| burger | sandwich, meat, bread | thiếu sandwich, bread |

**Vòng 1:** chỉ bread nấu được. Bây giờ kho có thêm `bread`.

**Vòng 2:** sandwich hết thiếu (bread đã có) → nấu. Burger vẫn thiếu sandwich (vì ta xét "cùng lúc").

**Vòng 3:** burger hết thiếu → nấu.

Quan sát quan trọng:
1. Thứ tự nấu chính là một **thứ tự tô-pô** (topological order) của đồ thị phụ thuộc.
2. Ta không cần biết *thiếu cái gì*, chỉ cần biết *còn thiếu bao nhiêu cái*. Đếm số thiếu = bậc vào còn lại.
3. Món nằm trong vòng tròn phụ thuộc (A cần B, B cần A) không bao giờ hết thiếu.
4. Món cần một nguyên liệu **không có trong supplies và cũng không phải món nào** (ví dụ "salt") sẽ thiếu mãi mãi.

Ta tính tay "số thiếu" cho ví dụ 3 bằng code mô phỏng từng vòng:

In [7]:
recipes, ingredients, supplies, _ = examples[2]
have = set(supplies)
done = set()
round_no = 0
while True:
    round_no += 1
    newly = [r for r, need in zip(recipes, ingredients)
             if r not in done and all(x in have for x in need)]
    missing = {r: [x for x in need if x not in have]
               for r, need in zip(recipes, ingredients) if r not in done}
    print(f"Vòng {round_no}: còn thiếu {missing} -> nấu được {newly}")
    if not newly:
        break
    for r in newly:
        done.add(r); have.add(r)
print("Kết quả:", sorted(done))

Vòng 1: còn thiếu {'bread': [], 'sandwich': ['bread'], 'burger': ['sandwich', 'bread']} -> nấu được ['bread']
Vòng 2: còn thiếu {'sandwich': [], 'burger': ['sandwich']} -> nấu được ['sandwich']
Vòng 3: còn thiếu {'burger': []} -> nấu được ['burger']
Vòng 4: còn thiếu {} -> nấu được []
Kết quả: ['bread', 'burger', 'sandwich']


## 5. Cách 1 — Brute Force / baseline: lặp đến khi không nấu thêm được

**Ý tưởng:** đúng như mô phỏng ở mục 4. Lặp lại: quét mọi món chưa nấu, món nào đủ nguyên liệu (trong tập `have`) thì nấu và thêm vào `have`. Khi một lượt quét không nấu thêm được món nào ⇒ dừng.

Mỗi lượt nấu thêm ít nhất 1 món (nếu không thì dừng), nên có tối đa `n + 1` lượt.

In [8]:
def brute_fixed_point(recipes, ingredients, supplies):
    have = set(supplies)
    done = []
    changed = True
    while changed:
        changed = False
        for r, need in zip(recipes, ingredients):
            if r not in have and all(x in have for x in need):
                have.add(r)          # nấu xong -> có thêm nguyên liệu r
                done.append(r)
                changed = True
    return done

for r, ing, sup, exp in examples:
    got = brute_fixed_point(r, ing, list(sup))
    print(got, "OK" if sorted(got) == sorted(exp) else "SAI")

['bread'] OK
['bread', 'sandwich'] OK
['bread', 'sandwich', 'burger'] OK


**Độ phức tạp:** gọi `n` = số món, `m` = tổng số nguyên liệu trong mọi công thức (`m ≤ 100·100 = 10⁴`).
- Mỗi lượt quét tốn O(m) (kiểm tra mọi nguyên liệu của mọi món).
- Số lượt tối đa O(n) — trường hợp xấu nhất là một **chuỗi** phụ thuộc được liệt kê **ngược** (món cuối của chuỗi đứng đầu danh sách), mỗi lượt chỉ nấu thêm đúng 1 món.
- Tổng: **O(n · m)** thời gian, O(n + |supplies|) bộ nhớ.

**Nút thắt:** ta kiểm tra đi kiểm tra lại cùng một món dù không có gì liên quan đến nó thay đổi. Ví dụ dưới đây cho thấy số lần kiểm tra nguyên liệu tăng theo bình phương khi chuỗi bị liệt kê ngược.

In [9]:
def make_chain(n):
    # r0 cần base, r1 cần r0, ..., liệt kê NGƯỢC để brute force tệ nhất
    recipes = [f"r{i}" for i in range(n)][::-1]
    ingredients = [["base"] if i == 0 else [f"r{i-1}"] for i in range(n)][::-1]
    return recipes, ingredients, ["base"]

def brute_count_checks(recipes, ingredients, supplies):
    have = set(supplies); checks = 0; changed = True
    while changed:
        changed = False
        for r, need in zip(recipes, ingredients):
            if r in have: continue
            checks += len(need)
            if all(x in have for x in need):
                have.add(r); changed = True
    return checks

for n in [10, 20, 40, 80]:
    print(n, "món -> số lần kiểm tra nguyên liệu:", brute_count_checks(*make_chain(n)))

10 món -> số lần kiểm tra nguyên liệu: 55
20 món -> số lần kiểm tra nguyên liệu: 210
40 món -> số lần kiểm tra nguyên liệu: 820
80 món -> số lần kiểm tra nguyên liệu: 3240


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** thay vì hỏi lại *"món này đủ chưa?"* cho mọi món ở mọi lượt, hãy **đảo chiều**: khi một thứ `x` vừa trở thành "đã có", chỉ báo tin cho **những món cần `x`**. Mỗi món giữ một bộ đếm `indeg[r]` = "còn thiếu bao nhiêu nguyên liệu".

Quy tắc "nếu… thì…":
- **Nếu** `x` vừa có (là supply ban đầu hoặc món vừa nấu) **thì** với mỗi món `r` trong `g[x]`: `indeg[r] -= 1`.
- **Nếu** `indeg[r]` vừa về `0` **thì** `r` nấu được → ghi vào đáp án và đưa `r` vào hàng đợi (vì giờ `r` cũng là một thứ "đã có").
- **Nếu** hàng đợi rỗng **thì** dừng: món nào còn `indeg > 0` là không nấu được.

**Bất biến (invariant):** tại mọi thời điểm, `indeg[r]` = số nguyên liệu của `r` **chưa được lấy ra khỏi hàng đợi**. Mỗi "thứ đã có" được đưa vào hàng đợi đúng một lần, nên mỗi cạnh `x → r` bị "tiêu thụ" đúng một lần ⇒ tổng công việc O(n + m + |supplies|).

**Vì sao chu trình tự động bị loại?** Trong chu trình `A → B → A`, `A` chỉ vào hàng đợi khi `indeg[A] = 0`, tức sau khi `B` đã vào; `B` lại cần `A` vào trước. Không ai đi trước ⇒ cả hai không bao giờ vào hàng đợi. Không cần code phát hiện chu trình riêng!

**Vì sao nguyên liệu "ma" (không có ở đâu) tự động bị loại?** Nó không bao giờ vào hàng đợi, nên cạnh từ nó không bao giờ bị tiêu thụ, `indeg` của món cần nó không bao giờ về 0.

Đây chính là **thuật toán Kahn**, với các nguồn (source) là `supplies`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS + memo + 3 màu

Định nghĩa `can(x)` = "có thể có `x` không?":
- `x` thuộc `supplies` ⇒ `True`.
- `x` không phải món nào ⇒ `False` (nguyên liệu ma).
- `x` là món: `can(x) = all(can(y) for y in ingredients của x)`.

Vì có thể có chu trình, ta đánh dấu `x` là **xám** trước khi đệ quy. Gặp lại đỉnh xám ⇒ trả `False` (đỉnh trong chu trình không thể nấu). Kết quả cuối ghi vào memo để mỗi món chỉ tính một lần.

**Tinh tế:** nếu `A` gặp `B` đang xám và trả `False`, thì `A` đúng là không nấu được? Có: `B` đang xám nghĩa là `B` (gián tiếp) cần `A`, mà `A` cần `B` ⇒ `A` nằm trong chu trình với `B` ⇒ cả hai không nấu được. Vì mỗi món chỉ có **một công thức** và cần **tất cả** nguyên liệu (logic "AND"), không có đường vòng nào khác.

In [10]:
def dfs_memo(recipes, ingredients, supplies):
    need = {r: ing for r, ing in zip(recipes, ingredients)}
    sup = set(supplies)
    GRAY = "gray"
    state = {}              # GRAY = đang thăm, True/False = đã có kết quả
    def can(x):
        if x in sup:
            return True
        if x not in need:   # nguyên liệu ma
            return False
        if x in state:
            st = state[x]
            return False if st == GRAY else st   # gặp xám -> chu trình -> False
        state[x] = GRAY
        ok = all(can(y) for y in need[x])
        state[x] = ok
        return ok
    return [r for r in recipes if can(r)]

for r, ing, sup, exp in examples:
    got = dfs_memo(r, ing, sup)
    print(got, "OK" if sorted(got) == sorted(exp) else "SAI")
print(dfs_memo(["a", "b", "c"], [["b"], ["a"], ["x"]], ["x"]))  # a,b chu trình -> chỉ c

['bread'] OK
['bread', 'sandwich'] OK
['bread', 'sandwich', 'burger'] OK
['c']


Lưu ý: ở `state[x] == 1` (gặp đỉnh xám), ta trả `False` cho lời gọi hiện tại nhưng **chưa** ghi memo cho các đỉnh khác. Đỉnh xám sau khi `all(...)` xong sẽ tự ghi `False` (vì có ít nhất một nguyên liệu trả `False`). Do `all` dừng sớm, có thể có nguyên liệu chưa được thăm — không sao, chúng sẽ được tính khi cần.

Code nộp LeetCode (cách 2). Độ sâu đệ quy tối đa ≤ n ≤ 100 nên an toàn với giới hạn đệ quy mặc định của Python (1000).

In [11]:
class SolutionDFS:
    def findAllRecipes(self, recipes: List[str], ingredients: List[List[str]], supplies: List[str]) -> List[str]:
        need = dict(zip(recipes, ingredients))
        sup = set(supplies)
        state = {}
        def can(x):
            if x in sup:
                return True
            if x not in need:
                return False
            if x in state:
                return state[x] is True
            state[x] = 1
            state[x] = all(can(y) for y in need[x])
            return state[x]
        return [r for r in recipes if can(r)]

print(SolutionDFS().findAllRecipes(*examples[2][:3]))

['bread', 'sandwich', 'burger']


Ở bản nộp, ta viết gọn `return state[x] is True`: nếu đang xám (`1`) thì `1 is True` là `False`; nếu đã có kết quả thì trả đúng kết quả. Chú ý phải dùng `is True` chứ **không** dùng `== True`, vì `1 == True` trong Python là `True` (bool là lớp con của int)!

**Độ phức tạp cách 2:** mỗi món tính một lần, mỗi cạnh duyệt tối đa một lần ⇒ **O(n + m + |supplies|)** thời gian, O(n + |supplies|) bộ nhớ (memo + ngăn xếp đệ quy).

In [12]:
gray = 1                       # trạng thái "xám" được mã hóa bằng số 1
print("gray == True :", gray == True)
print("gray is True :", gray is True)

gray == True : True
gray is True : False


### 7.2 Cách 3 — Thuật toán Kahn (BFS topological sort) — cách chuẩn

Ba bước:
1. **Xây đồ thị:** với mỗi món `r` và mỗi nguyên liệu `x` của nó, thêm cạnh `x → r`; đặt `indeg[r] = len(ingredients)`.
2. **Khởi tạo hàng đợi** bằng toàn bộ `supplies`.
3. **BFS:** lấy `x` ra, giảm `indeg` mọi món trong `g[x]`; món nào về 0 thì ghi vào đáp án và đẩy vào hàng đợi.

In [13]:
def kahn(recipes, ingredients, supplies):
    g = defaultdict(list)
    indeg = {}
    for r, need in zip(recipes, ingredients):
        indeg[r] = len(need)
        for x in need:
            g[x].append(r)
    q = deque(supplies)
    ans = []
    while q:
        x = q.popleft()
        for r in g.get(x, []):
            indeg[r] -= 1
            if indeg[r] == 0:
                ans.append(r)
                q.append(r)
    return ans

for r, ing, sup, exp in examples:
    got = kahn(r, ing, sup)
    print(got, "OK" if sorted(got) == sorted(exp) else "SAI")

['bread'] OK
['bread', 'sandwich'] OK
['bread', 'sandwich', 'burger'] OK


Code nộp LeetCode (cách 3), dùng mẹo duyệt list đang được nối thêm (giống doocs) — lưu ý ta **sao chép** `supplies` bằng `list(supplies)` để không sửa input của người gọi:

In [14]:
class SolutionKahn:
    def findAllRecipes(self, recipes: List[str], ingredients: List[List[str]], supplies: List[str]) -> List[str]:
        g = defaultdict(list)
        indeg = {}
        for r, need in zip(recipes, ingredients):
            indeg[r] = len(need)
            for x in need:
                g[x].append(r)
        q = list(supplies)
        ans = []
        for x in q:                 # q dài thêm trong lúc duyệt
            for r in g.get(x, ()):
                indeg[r] -= 1
                if indeg[r] == 0:
                    ans.append(r)
                    q.append(r)
        return ans

print(SolutionKahn().findAllRecipes(*examples[2][:3]))

['bread', 'sandwich', 'burger']


**Độ phức tạp cách 3:** xây đồ thị O(m); mỗi phần tử vào hàng đợi tối đa một lần (supplies: |S| lần, món: ≤ n lần); mỗi cạnh giảm bậc đúng một lần ⇒ **O(n + m + |S|)** thời gian, O(n + m + |S|) bộ nhớ cho đồ thị.

### 7.3 Cách 4 — Kahn "chỉ trên món ăn" (đếm số nguyên liệu *chưa có sẵn*)

Biến thể nhỏ: không cho supplies vào hàng đợi. Với mỗi món, chỉ đếm những nguyên liệu **không** nằm trong `supplies`. Nguyên liệu không có trong supplies:
- là một món khác ⇒ thêm cạnh `món đó → r`;
- không là món nào ⇒ `r` chắc chắn không nấu được (đánh dấu `bad`).

Khởi tạo hàng đợi bằng các món có bậc vào 0. Cách này cho đồ thị nhỏ hơn (chỉ ≤ n đỉnh) và loại sớm món cần "nguyên liệu ma".

In [15]:
def kahn_recipes_only(recipes, ingredients, supplies):
    sup = set(supplies)
    is_recipe = set(recipes)
    g = defaultdict(list)
    indeg = {}
    for r, need in zip(recipes, ingredients):
        cnt = 0
        for x in need:
            if x in sup:
                continue
            if x not in is_recipe:
                cnt = float("inf")     # nguyên liệu ma -> không bao giờ về 0
                break
            g[x].append(r)
            cnt += 1
        indeg[r] = cnt
    q = deque(r for r in recipes if indeg[r] == 0)
    ans = []
    while q:
        r = q.popleft()
        ans.append(r)
        for nxt in g[r]:
            indeg[nxt] -= 1
            if indeg[nxt] == 0:
                q.append(nxt)
    return ans

for r, ing, sup, exp in examples:
    got = kahn_recipes_only(r, ing, sup)
    print(got, "OK" if sorted(got) == sorted(exp) else "SAI")
print(kahn_recipes_only(["a", "b"], [["salt"], ["a", "x"]], ["x"]))  # a cần salt (ma) -> []

['bread'] OK
['bread', 'sandwich'] OK
['bread', 'sandwich', 'burger'] OK
[]


Cẩn thận ở dòng `break` khi gặp nguyên liệu ma: các cạnh của những nguyên liệu **đã duyệt trước đó** đã được thêm vào `g`, nên sau này `indeg[r]` (đang là `inf`) có thể bị trừ — `inf - 1` vẫn là `inf`, không bao giờ bằng 0, nên vẫn đúng. Nếu muốn sạch hơn, có thể kiểm tra "có nguyên liệu ma không" trước rồi mới thêm cạnh.

**Độ phức tạp cách 4:** O(n + m + |S|) (tạo `set(supplies)`), bộ nhớ đồ thị chỉ gồm cạnh giữa các món.

## 8. TRACE từng bước

Ta trace thuật toán Kahn (cách 3) trên ví dụ 3. Bảng trạng thái mong đợi:

| Bước | Lấy ra | Món bị giảm bậc | indeg sau bước (bread, sandwich, burger) | Đáp án |
|---|---|---|---|---|
| 0 | — | — | (2, 2, 3) | [] |
| 1 | yeast | bread | (1, 2, 3) | [] |
| 2 | flour | bread → 0 | (0, 2, 3) | [bread] |
| 3 | meat | sandwich, burger | (0, 1, 2) | [bread] |
| 4 | bread | sandwich → 0, burger | (0, 0, 1) | [bread, sandwich] |
| 5 | sandwich | burger → 0 | (0, 0, 0) | [bread, sandwich, burger] |
| 6 | burger | (không ai cần) | (0, 0, 0) | [bread, sandwich, burger] |

In [16]:
def trace_kahn(recipes, ingredients, supplies):
    g = defaultdict(list); indeg = {}
    for r, need in zip(recipes, ingredients):
        indeg[r] = len(need)
        for x in need:
            g[x].append(r)
    print("đồ thị:", dict(g))
    print("indeg ban đầu:", indeg)
    q = deque(supplies); ans = []; step = 0
    while q:
        x = q.popleft(); step += 1
        touched = []
        for r in g.get(x, []):
            indeg[r] -= 1
            touched.append(f"{r}->{indeg[r]}")
            if indeg[r] == 0:
                ans.append(r); q.append(r)
        print(f"bước {step}: lấy '{x}', giảm [{', '.join(touched)}], hàng đợi={list(q)}, ans={ans}")
    return ans

trace_kahn(*examples[2][:3])

đồ thị: {'yeast': ['bread'], 'flour': ['bread'], 'bread': ['sandwich', 'burger'], 'meat': ['sandwich', 'burger'], 'sandwich': ['burger']}
indeg ban đầu: {'bread': 2, 'sandwich': 2, 'burger': 3}
bước 1: lấy 'yeast', giảm [bread->1], hàng đợi=['flour', 'meat'], ans=[]
bước 2: lấy 'flour', giảm [bread->0], hàng đợi=['meat', 'bread'], ans=['bread']
bước 3: lấy 'meat', giảm [sandwich->1, burger->2], hàng đợi=['bread'], ans=['bread']
bước 4: lấy 'bread', giảm [sandwich->0, burger->1], hàng đợi=['sandwich'], ans=['bread', 'sandwich']
bước 5: lấy 'sandwich', giảm [burger->0], hàng đợi=['burger'], ans=['bread', 'sandwich', 'burger']
bước 6: lấy 'burger', giảm [], hàng đợi=[], ans=['bread', 'sandwich', 'burger']


['bread', 'sandwich', 'burger']

Trace thêm một ví dụ có **chu trình** và **nguyên liệu ma** để thấy các món đó "kẹt" với `indeg > 0`:
- `a` cần `b`, `b` cần `a` (chu trình).
- `c` cần `salt` (không có ở đâu).
- `d` cần `x` (có sẵn).

In [17]:
res = trace_kahn(["a", "b", "c", "d"], [["b"], ["a"], ["salt"], ["x"]], ["x"])
print("Kết quả:", res)

đồ thị: {'b': ['a'], 'a': ['b'], 'salt': ['c'], 'x': ['d']}
indeg ban đầu: {'a': 1, 'b': 1, 'c': 1, 'd': 1}
bước 1: lấy 'x', giảm [d->0], hàng đợi=['d'], ans=['d']
bước 2: lấy 'd', giảm [], hàng đợi=[], ans=['d']
Kết quả: ['d']


Trace cách 2 (DFS) để thấy trạng thái màu thay đổi thế nào khi gặp chu trình:

In [18]:
def trace_dfs(recipes, ingredients, supplies):
    need = dict(zip(recipes, ingredients)); sup = set(supplies); state = {}
    def can(x, depth):
        pad = "  " * depth
        if x in sup:
            print(f"{pad}{x}: có sẵn -> True"); return True
        if x not in need:
            print(f"{pad}{x}: nguyên liệu ma -> False"); return False
        if x in state:
            v = state[x] is True
            print(f"{pad}{x}: đã có trạng thái {state[x]!r} -> {v}"); return v
        print(f"{pad}{x}: tô XÁM, xét {need[x]}")
        state[x] = 1
        state[x] = all(can(y, depth + 1) for y in need[x])
        print(f"{pad}{x}: tô ĐEN, kết quả {state[x]}")
        return state[x]
    return [r for r in recipes if can(r, 0)]

print(trace_dfs(["a", "b", "d"], [["b", "x"], ["a"], ["x"]], ["x"]))

a: tô XÁM, xét ['b', 'x']
  b: tô XÁM, xét ['a']
    a: đã có trạng thái 1 -> False
  b: tô ĐEN, kết quả False
a: tô ĐEN, kết quả False
b: đã có trạng thái False -> False
d: tô XÁM, xét ['x']
  x: có sẵn -> True
d: tô ĐEN, kết quả True
['d']


## 9. Code nộp LeetCode

Bản chuẩn: thuật toán Kahn với `deque`. Trên LeetCode, `List`, `defaultdict`, `deque` đã được import sẵn; ở đây ta import rõ ràng để notebook tự chạy được.

In [19]:
from typing import List
from collections import defaultdict, deque

class Solution:
    def findAllRecipes(self, recipes: List[str], ingredients: List[List[str]], supplies: List[str]) -> List[str]:
        g = defaultdict(list)          # nguyên liệu -> các món cần nó
        indeg = {}                     # món -> số nguyên liệu còn thiếu
        for r, need in zip(recipes, ingredients):
            indeg[r] = len(need)
            for x in need:
                g[x].append(r)
        q = deque(supplies)
        ans = []
        while q:
            x = q.popleft()
            for r in g.get(x, ()):
                indeg[r] -= 1
                if indeg[r] == 0:
                    ans.append(r)
                    q.append(r)
        return ans

In [20]:
sol = Solution()
for r, ing, sup, exp in examples:
    got = sol.findAllRecipes(r, ing, sup)
    print(got, "| kỳ vọng", exp, "| OK" if sorted(got) == sorted(exp) else "| SAI")

['bread'] | kỳ vọng ['bread'] | OK
['bread', 'sandwich'] | kỳ vọng ['bread', 'sandwich'] | OK
['bread', 'sandwich', 'burger'] | kỳ vọng ['bread', 'sandwich', 'burger'] | OK


## 10. Vì sao thuật toán đúng

Gọi một thứ `x` là **đạt được** nếu `x ∈ supplies`, hoặc `x` là món mà mọi nguyên liệu của nó đều đạt được — định nghĩa theo nghĩa "có thể dựng được bằng một chuỗi nấu hữu hạn". Đáp án đúng = tập món đạt được.

**(a) Mọi món Kahn trả về đều đạt được (tính đúng đắn — soundness).** Quy nạp theo thứ tự vào hàng đợi. Supplies đạt được. Một món `r` vào hàng đợi khi `indeg[r] = 0`, tức **mọi** nguyên liệu của `r` đã được lấy ra khỏi hàng đợi trước đó (mỗi lần lấy ra một nguyên liệu trừ đúng 1; `ingredients[i]` không có phần tử trùng nên đếm `len` là chính xác). Theo giả thiết quy nạp chúng đều đạt được ⇒ `r` đạt được.

**(b) Mọi món đạt được đều được Kahn trả về (tính đầy đủ — completeness).** Món đạt được có một "độ sâu" `d(r)` = số vòng nấu cần thiết (như mục 4). Quy nạp theo `d`: mọi nguyên liệu của `r` có độ sâu nhỏ hơn ⇒ theo quy nạp đều vào hàng đợi ⇒ đều được lấy ra (hàng đợi chỉ dừng khi rỗng) ⇒ `indeg[r]` bị trừ đủ `len(need)` lần ⇒ về 0 ⇒ `r` vào đáp án.

**(c) Không trùng lặp:** `indeg[r]` chỉ giảm, nên chỉ **một lần** đi qua giá trị 0 ⇒ mỗi món được thêm tối đa một lần. Tên recipes và supplies phân biệt nên không món nào "vừa có sẵn vừa được nấu".

**(d) Dừng:** mỗi phần tử vào hàng đợi tối đa một lần, số phần tử hữu hạn.

Với DFS (cách 2): `can(x)` chính là định nghĩa đệ quy của "đạt được"; trạng thái xám xử lý trường hợp định nghĩa đệ quy tự tham chiếu — món trong chu trình không có chuỗi nấu hữu hạn nên phải là `False`.

## 11. Phân tích Time / Space Complexity

Ký hiệu: `n` = số món, `m` = tổng `len(ingredients[i])` (≤ 10⁴), `S` = `len(supplies)` (≤ 100). Độ dài chuỗi ≤ 10 nên băm chuỗi coi như O(1).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Lặp đến ổn định | O(n · m) worst | O(n + S) | tệ khi chuỗi phụ thuộc liệt kê ngược |
| 2. DFS + memo | O(n + m + S) | O(n + S) + ngăn xếp O(n) | đệ quy sâu tối đa n |
| 3. Kahn | O(n + m + S) | O(n + m + S) | chuẩn, không đệ quy |
| 4. Kahn chỉ trên món | O(n + m + S) | O(n + m) | đồ thị nhỏ hơn |

Với ràng buộc của đề (n ≤ 100), cả 4 cách đều chạy nhanh; nhưng Kahn là cách **mở rộng tốt** nếu n lên 10⁵. Ta đo nhanh trên chuỗi dài:

In [21]:
for n in [200, 800]:
    rs, ings, sup = make_chain(n)
    t = time.perf_counter(); brute_fixed_point(rs, ings, sup); tb = time.perf_counter() - t
    t = time.perf_counter(); kahn(rs, ings, sup); tk = time.perf_counter() - t
    print(f"n={n}: brute {tb*1000:.1f} ms, kahn {tk*1000:.2f} ms")

n=200: brute 5.5 ms, kahn 0.14 ms


n=800: brute 64.1 ms, kahn 0.71 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case thủ công

In [22]:
ALL = {
    "brute": brute_fixed_point,
    "dfs": dfs_memo,
    "kahn": kahn,
    "kahn_recipes": kahn_recipes_only,
    "Solution": lambda r, i, s: Solution().findAllRecipes(r, i, s),
    "SolutionDFS": lambda r, i, s: SolutionDFS().findAllRecipes(r, i, s),
    "SolutionKahn": lambda r, i, s: SolutionKahn().findAllRecipes(r, i, s),
}
cases = [c[:4] for c in examples] + [
    (["a"], [["x"]], ["y"], []),                                  # thiếu nguyên liệu
    (["a", "b"], [["b"], ["a"]], ["x"], []),                      # chu trình 2
    (["a", "b", "c"], [["b"], ["c"], ["a"]], ["x"], []),          # chu trình 3
    (["a", "b"], [["b", "x"], ["a", "x"]], ["x"], []),            # chu trình dù có x
    (["c", "b", "a"], [["b"], ["a"], ["x"]], ["x"], ["a", "b", "c"]),  # chuỗi liệt kê ngược
    (["a", "b"], [["x"], ["a", "salt"]], ["x"], ["a"]),           # b cần nguyên liệu ma
    (["a", "b", "c"], [["x"], ["a"], ["a", "b"]], ["x"], ["a", "b", "c"]),
    (["a", "b", "c"], [["b"], ["a"], ["x"]], ["x"], ["c"]),       # chu trình + món độc lập
]
for r, ing, sup, exp in cases:
    for name, f in ALL.items():
        got = f(list(r), [list(v) for v in ing], list(sup))
        assert sorted(got) == sorted(exp), (name, r, ing, sup, got, exp)
        assert len(got) == len(set(got)), (name, "trùng lặp")
print("Tất cả", len(cases), "case x", len(ALL), "cách: OK")

Tất cả 11 case x 7 cách: OK


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Sinh đồ thị ngẫu nhiên đúng ràng buộc: tên món và supplies phân biệt, mỗi công thức không trùng nguyên liệu, có thể có chu trình và nguyên liệu ma. So sánh mọi cách với baseline.

In [23]:
def gen_case(rng):
    n = rng.randint(1, 12)
    names = [f"r{i}" for i in range(n)]
    sups = [f"s{i}" for i in range(rng.randint(1, 5))]
    ghosts = [f"g{i}" for i in range(rng.randint(0, 2))]
    pool = names + sups + ghosts
    ings = []
    for i in range(n):
        k = rng.randint(1, min(4, len(pool)))
        ings.append(rng.sample(pool, k))
    order = names[:]; rng.shuffle(order)
    idx = {r: i for i, r in enumerate(names)}
    return order, [ings[idx[r]] for r in order], sups

rng = random.Random(2115)
cnt_nonempty = 0
for _ in range(3000):
    r, ing, sup = gen_case(rng)
    ref = sorted(brute_fixed_point(r, ing, sup))
    cnt_nonempty += bool(ref)
    for name, f in ALL.items():
        got = f(list(r), [list(v) for v in ing], list(sup))
        assert sorted(got) == ref, (name, r, ing, sup, got, ref)
print("3000 case ngẫu nhiên OK; số case có đáp án khác rỗng:", cnt_nonempty)

3000 case ngẫu nhiên OK; số case có đáp án khác rỗng: 1314


### 12.3 Test kích thước lớn nhất

`n = 100`, mỗi món 100 nguyên liệu — chạy nhanh với mọi cách.

In [24]:
rng = random.Random(7)
names = [f"r{i}" for i in range(100)]
sups = [f"s{i}" for i in range(100)]
ings = []
for i in range(100):
    pool = sups + names[:i]          # chỉ phụ thuộc món trước -> DAG, tất cả nấu được
    ings.append(rng.sample(pool, min(100, len(pool))))
t = time.perf_counter()
out = Solution().findAllRecipes(names, ings, sups)
print(len(out), "món,", f"{(time.perf_counter()-t)*1000:.2f} ms")
assert sorted(out) == sorted(names)
assert sorted(dfs_memo(names, ings, sups)) == sorted(names)

100 món, 1.00 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quét một lượt duy nhất theo thứ tự input

Người mới hay nghĩ: "duyệt từng món, đủ thì nấu". Nhưng nếu món phụ thuộc đứng **trước** món nó cần trong danh sách, một lượt là không đủ.

In [25]:
def wrong_one_pass(recipes, ingredients, supplies):
    have = set(supplies); ans = []
    for r, need in zip(recipes, ingredients):
        if all(x in have for x in need):
            have.add(r); ans.append(r)
    return ans

r, ing, sup = ["sandwich", "bread"], [["bread", "meat"], ["yeast", "flour"]], ["yeast", "flour", "meat"]
print("bản sai :", wrong_one_pass(r, ing, sup))
print("bản đúng:", kahn(r, ing, sup))

bản sai : ['bread']
bản đúng: ['bread', 'sandwich']


**Vì sao sai:** khi xét sandwich, bread chưa được nấu. Sau đó bread nấu được nhưng ta không quay lại sandwich. **Sửa:** lặp đến khi ổn định (cách 1) hoặc dùng Kahn để "báo tin" cho các món phụ thuộc.

### Lỗi 2 — DFS không có trạng thái "đang thăm" → đệ quy vô hạn

Không đánh dấu trước khi đệ quy thì chu trình `a ↔ b` làm `can(a) → can(b) → can(a) → …` cho tới khi `RecursionError`.

In [26]:
import sys
def wrong_dfs_no_gray(recipes, ingredients, supplies):
    need = dict(zip(recipes, ingredients)); sup = set(supplies); memo = {}
    def can(x):
        if x in sup: return True
        if x not in need: return False
        if x in memo: return memo[x]
        memo[x] = all(can(y) for y in need[x])   # chỉ ghi memo SAU khi đệ quy
        return memo[x]
    return [r for r in recipes if can(r)]

old = sys.getrecursionlimit()
sys.setrecursionlimit(300)       # hạ giới hạn để lỗi xuất hiện nhanh
try:
    print(wrong_dfs_no_gray(["a", "b"], [["b"], ["a"]], ["x"]))
except RecursionError as e:
    print("bản sai -> RecursionError:", str(e)[:40])
finally:
    sys.setrecursionlimit(old)
print("bản đúng:", dfs_memo(["a", "b"], [["b"], ["a"]], ["x"]))

bản sai -> RecursionError: maximum recursion depth exceeded
bản đúng: []


**Sửa:** gán `state[x] = 1` (xám) **trước** khi đệ quy, và khi gặp lại đỉnh xám thì trả `False`.

### Lỗi 3 — Coi đỉnh "đang thăm" là `True` (lạc quan sai)

Một số bạn đặt `memo[x] = True` trước khi đệ quy để "chặn" chu trình. Không lặp vô hạn nữa, nhưng kết quả sai: chu trình được coi là nấu được!

In [27]:
def wrong_dfs_optimistic(recipes, ingredients, supplies):
    need = dict(zip(recipes, ingredients)); sup = set(supplies); memo = {}
    def can(x):
        if x in sup: return True
        if x not in need: return False
        if x in memo: return memo[x]
        memo[x] = True                          # SAI: giả định nấu được
        memo[x] = all(can(y) for y in need[x])
        return memo[x]
    return [r for r in recipes if can(r)]

r, ing, sup = ["a", "b"], [["b", "x"], ["a", "x"]], ["x"]
print("bản sai :", wrong_dfs_optimistic(r, ing, sup))
print("bản đúng:", dfs_memo(r, ing, sup))

bản sai : ['a', 'b']
bản đúng: []


**Vì sao sai:** `can(a)` đặt `a = True`, gọi `can(b)`, `b` cần `a` → thấy `True` → `b = True` → `a = True`. Một vòng tròn "tự chứng minh" chính nó. **Sửa:** đỉnh xám phải trả `False`.

### Lỗi 4 — Dùng `== True` thay vì `is True` khi trạng thái xám là số 1

In [28]:
def wrong_eq_true(recipes, ingredients, supplies):
    need = dict(zip(recipes, ingredients)); sup = set(supplies); state = {}
    def can(x):
        if x in sup: return True
        if x not in need: return False
        if x in state: return state[x] == True     # SAI: 1 == True là True!
        state[x] = 1
        state[x] = all(can(y) for y in need[x])
        return state[x]
    return [r for r in recipes if can(r)]

print("bản sai :", wrong_eq_true(r, ing, sup))
print("bản đúng:", SolutionDFS().findAllRecipes(r, ing, sup))

bản sai : ['a', 'b']
bản đúng: []


**Sửa:** dùng `state[x] is True`, hoặc tốt hơn dùng hằng số rõ ràng (`GRAY = 0, YES = 1, NO = 2`) để không lẫn với bool.

### Lỗi 5 — Kahn nhưng khởi tạo bậc vào sai (đếm cả nguyên liệu có sẵn hai lần / quên supplies)

Ví dụ: khởi tạo hàng đợi bằng các món có `indeg == 0` (theo kiểu topo sort trên đồ thị món) nhưng `indeg` lại tính **cả** supplies — thì không món nào có bậc 0, đáp án luôn rỗng.

In [29]:
def wrong_kahn_no_supplies(recipes, ingredients, supplies):
    g = defaultdict(list); indeg = {}
    for r, need in zip(recipes, ingredients):
        indeg[r] = len(need)
        for x in need: g[x].append(r)
    q = deque(r for r in recipes if indeg[r] == 0)   # SAI: quên đưa supplies vào
    ans = []
    while q:
        x = q.popleft()
        for r in g[x]:
            indeg[r] -= 1
            if indeg[r] == 0: ans.append(r); q.append(r)
    return ans

print("bản sai :", wrong_kahn_no_supplies(*examples[1][:3]))
print("bản đúng:", kahn(*examples[1][:3]))

bản sai : []
bản đúng: ['bread', 'sandwich']


**Sửa:** hoặc đưa supplies vào hàng đợi (cách 3), hoặc khi tính bậc vào thì **bỏ qua** nguyên liệu có sẵn (cách 4). Hai cách nhất quán với nhau — không được trộn nửa nọ nửa kia.

### Lỗi 6 — Sửa trực tiếp `supplies` của người gọi

Bản doocs viết `q = supplies` rồi `q.append(...)` — trên LeetCode không sao, nhưng trong code thật nó làm thay đổi list đầu vào, gây lỗi khó tìm nếu caller dùng lại.

In [30]:
def kahn_alias(recipes, ingredients, supplies):
    g = defaultdict(list); indeg = {}
    for r, need in zip(recipes, ingredients):
        indeg[r] = len(need)
        for x in need: g[x].append(r)
    q = supplies                 # cùng một object!
    ans = []
    for x in q:
        for r in g[x]:
            indeg[r] -= 1
            if indeg[r] == 0: ans.append(r); q.append(r)
    return ans

sup = ["yeast", "flour", "meat"]
kahn_alias(["bread", "sandwich"], [["yeast", "flour"], ["bread", "meat"]], sup)
print("supplies sau khi gọi:", sup)

supplies sau khi gọi: ['yeast', 'flour', 'meat', 'bread', 'sandwich']


**Sửa:** `q = list(supplies)` hoặc `deque(supplies)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Dấu hiệu topo sort:** "A cần B", "làm X trước Y", "phụ thuộc", "prerequisite", "có thể có vòng". → nghĩ ngay tới Kahn (bậc vào) hoặc DFS 3 màu.
- **Logic AND** (cần *tất cả* nguyên liệu) ⇒ bậc vào = số điều kiện; về 0 mới "mở khóa". Nếu là **logic OR** (cần *một trong*) thì chỉ cần BFS thường: chạm tới là mở.
- **Nhiều nguồn (multi-source):** supplies là các nguồn → khởi tạo hàng đợi bằng **tất cả** chúng cùng lúc.
- Chu trình **tự loại** trong Kahn: không cần code phát hiện riêng. Muốn biết "có chu trình không" thì so số đỉnh đã xử lý với tổng số đỉnh.
- Đỉnh là chuỗi → dùng `dict`/`defaultdict` thay vì mảng chỉ số. Nếu muốn nhanh hơn, ánh xạ tên → id số nguyên.
- Bài cùng họ: 207/210 Course Schedule I/II, 269 Alien Dictionary, 1136 Parallel Courses, 2392 Build a Matrix With Conditions, 802 Find Eventual Safe States.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trả về **số vòng nấu** tối thiểu (mỗi vòng nấu song song mọi món đã đủ nguyên liệu)

Dùng Kahn theo **tầng** (level-order BFS): `level[r] = 1 + max(level của nguyên liệu)`, supplies có level 0. Đáp án là level lớn nhất (0 nếu không nấu được gì).

In [31]:
def min_rounds(recipes, ingredients, supplies):
    g = defaultdict(list); indeg = {}
    for r, need in zip(recipes, ingredients):
        indeg[r] = len(need)
        for x in need: g[x].append(r)
    level = {s: 0 for s in supplies}
    q = deque(supplies); best = 0
    while q:
        x = q.popleft()
        for r in g.get(x, ()):
            level[r] = max(level.get(r, 0), level[x] + 1)
            indeg[r] -= 1
            if indeg[r] == 0:
                best = max(best, level[r]); q.append(r)
    return best

print(min_rounds(*examples[2][:3]))     # bread(1) -> sandwich(2) -> burger(3)
print(min_rounds(*examples[0][:3]))

3
1


Vì sao `level[r]` đúng khi `indeg[r]` về 0? Lúc đó mọi nguyên liệu của `r` đã được lấy ra và mỗi lần lấy ra đều cập nhật `max` ⇒ `level[r] = 1 + max(level nguyên liệu)` chính xác.

### 15.2 Biến thể B — Mua thêm **một** nguyên liệu để mở khóa nhiều món nhất

Cho phép mua thêm đúng 1 nguyên liệu (chưa có trong supplies, không phải món). Chọn nguyên liệu nào để số món nấu được là lớn nhất? Với n ≤ 100, thử từng ứng viên và chạy Kahn: O(K · (n + m)).

In [32]:
def best_extra(recipes, ingredients, supplies):
    rset, sset = set(recipes), set(supplies)
    cands = sorted({x for need in ingredients for x in need if x not in rset and x not in sset})
    base = len(kahn(recipes, ingredients, supplies))
    best = (base, None)
    for c in cands:
        k = len(kahn(recipes, ingredients, supplies + [c]))
        if k > best[0]:
            best = (k, c)
    return best

r = ["cake", "icing", "bread", "toast"]
ing = [["flour", "egg", "sugar"], ["sugar", "butter"], ["flour", "yeast"], ["bread", "butter"]]
print(best_extra(r, ing, ["flour", "egg", "yeast"]))

(2, 'butter')


### 15.3 Biến thể C — Logic OR: mỗi món có **nhiều công thức**, nấu được nếu đủ một công thức bất kỳ

Khi đó mỗi công thức là một "nút AND" riêng (bậc vào = số nguyên liệu của công thức), còn món là "nút OR" (mở khi **một** công thức của nó mở). Kahn vẫn dùng được: bậc vào đặt trên công thức, món được thêm vào hàng đợi lần đầu tiên một công thức của nó mở.

In [33]:
def recipes_with_alternatives(recipe_options, supplies):
    # recipe_options: dict món -> list các công thức (mỗi công thức là list nguyên liệu)
    g = defaultdict(list); indeg = []; owner = []
    for r, opts in recipe_options.items():
        for need in opts:
            cid = len(indeg); indeg.append(len(need)); owner.append(r)
            for x in need: g[x].append(cid)
    have = set(supplies); q = deque(supplies); ans = []
    while q:
        x = q.popleft()
        for cid in g.get(x, ()):
            indeg[cid] -= 1
            if indeg[cid] == 0 and owner[cid] not in have:
                have.add(owner[cid]); ans.append(owner[cid]); q.append(owner[cid])
    return ans

opts = {"bread": [["yeast", "flour"], ["flour", "soda"]], "toast": [["bread"]], "jam": [["fruit"], ["toast", "sugar"]]}
print(recipes_with_alternatives(opts, ["flour", "soda", "sugar"]))

['bread', 'toast', 'jam']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lặp đến ổn định | quét mọi món đến khi không đổi | O(n·m) | O(n + S) | n nhỏ, cần code nhanh, dễ đúng |
| 2. DFS + memo + 3 màu | định nghĩa đệ quy "có thể có x" | O(n + m + S) | O(n + S) + stack | khi chỉ cần hỏi vài món, thích tư duy đệ quy |
| 3. Kahn từ supplies | báo tin cho món phụ thuộc, bậc vào về 0 thì nấu | O(n + m + S) | O(n + m + S) | **mặc định**, không đệ quy, dễ mở rộng (level, đếm) |
| 4. Kahn chỉ trên món | bỏ qua supplies khi đếm bậc, loại sớm nguyên liệu ma | O(n + m + S) | O(n + m) | khi supplies rất lớn, muốn đồ thị nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `recipes = ["a","b","c"]`, `ingredients = [["b"],["c"],["x"]]`, `supplies = ["x"]`. Kết quả là gì? Nếu đổi `ingredients[2] = ["a"]` thì sao?

**Bài 2 (tính tay):** Với ví dụ 3, sau khi xây đồ thị, `g["bread"]` gồm những món nào và `indeg` ban đầu của mỗi món là bao nhiêu? Món nào được thêm vào đáp án khi lấy `"bread"` ra khỏi hàng đợi?

**Bài 3 (điền TODO):** Viết `bt3_todo(recipes, ingredients, supplies)` trả về **danh sách các món KHÔNG nấu được**, sắp xếp tăng dần theo tên.

**Bài 4 (tự viết lại):** Viết `bt4_todo(...)` cài đặt lại cách DFS 3 màu nhưng **không dùng đệ quy** (dùng ngăn xếp tường minh). Trả về danh sách món nấu được theo thứ tự xuất hiện trong `recipes`.

**Bài 5 (mini challenge):** Viết `bt5_todo(recipes, ingredients, supplies)` trả về **số nguyên liệu gốc tối thiểu** (thuộc `supplies`) mà món `recipes[0]` cần dùng tới (trực tiếp hoặc gián tiếp, đếm theo tập — mỗi nguyên liệu gốc đếm 1 lần). Nếu `recipes[0]` không nấu được, trả `-1`.

In [34]:
def bt3_todo(recipes, ingredients, supplies):
    # TODO: trả về sorted list các món không nấu được
    pass

def bt4_todo(recipes, ingredients, supplies):
    # TODO: DFS 3 màu bằng stack tường minh, không đệ quy
    raise NotImplementedError

def bt5_todo(recipes, ingredients, supplies):
    # TODO: số nguyên liệu gốc (supplies) khác nhau mà recipes[0] cần, hoặc -1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: vẽ cạnh. Mức 2: `c` cần `x` có sẵn; `b` cần `c`; `a` cần `b`. Mức 3: lần đầu là `[c, b, a]` (theo thứ tự nấu); khi đổi thành `c` cần `a` thì có chu trình `a→b→c→a` ⇒ `[]`.

**Bài 2.** Mức 1: `g[x]` = các món có `x` trong công thức. Mức 2: sandwich và burger đều có bread. Mức 3: `g["bread"] = ["sandwich","burger"]`; indeg = bread 2, sandwich 2, burger 3; lấy bread ra thì sandwich về 0 (đã trừ meat trước) ⇒ thêm sandwich.

**Bài 3.** Mức 1: lấy tập món nấu được bằng Kahn. Mức 2: hiệu tập `set(recipes) - set(ok)`. Mức 3: `sorted(set(recipes) - set(kahn(...)))`.

**Bài 4.** Mức 1: ngăn xếp chứa cặp `(món, chỉ số nguyên liệu đang xét)`. Mức 2: khi xét nguyên liệu `y`: có sẵn ⇒ sang cái kế; ma hoặc xám hoặc đã `False` ⇒ món hiện tại `False`, pop; chưa thăm ⇒ tô xám, push. Mức 3: khi một món pop với kết quả, "báo" kết quả lên phần tử dưới nó trên ngăn xếp (nếu `False` thì phần tử dưới cũng `False` và pop tiếp; nếu `True` thì tăng chỉ số).

**Bài 5.** Mức 1: trước hết kiểm tra `recipes[0]` có nấu được không. Mức 2: nếu nấu được thì mọi món trên đường đi đều nấu được, chỉ cần DFS/BFS gom tập supplies chạm tới. Mức 3: BFS từ `recipes[0]` theo chiều "cần" (`need[r]`), gom các `x ∈ supplies` vào một set, đánh dấu món đã thăm.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `["c","b","a"]` (thứ tự tùy ý: tập {a, b, c}); sau khi đổi ⇒ `[]`.

**Bài 2:** `g["bread"] = ["sandwich", "burger"]`, indeg ban đầu `{bread: 2, sandwich: 2, burger: 3}`; lấy bread ra (bước 4 trong trace) thì sandwich về 0 ⇒ được thêm vào đáp án, burger còn 1.

In [35]:
# Bài 1 & 2 kiểm chứng bằng code
assert sorted(kahn(["a","b","c"], [["b"],["c"],["x"]], ["x"])) == ["a","b","c"]
assert kahn(["a","b","c"], [["b"],["c"],["a"]], ["x"]) == []
g2 = defaultdict(list); ind2 = {}
for r, need in zip(*examples[2][:2]):
    ind2[r] = len(need)
    for x in need: g2[x].append(r)
assert g2["bread"] == ["sandwich", "burger"] and ind2 == {"bread": 2, "sandwich": 2, "burger": 3}

def bt3(recipes, ingredients, supplies):
    return sorted(set(recipes) - set(kahn(recipes, ingredients, supplies)))

def bt4(recipes, ingredients, supplies):
    need = dict(zip(recipes, ingredients)); sup = set(supplies)
    res = {}                         # món -> True/False; GRAY = "g"
    for start in recipes:
        if start in res: continue
        res[start] = "g"; stack = [[start, 0]]
        while stack:
            top = stack[-1]; r, i = top
            if i == len(need[r]):    # mọi nguyên liệu OK
                res[r] = True; stack.pop()
                if stack: stack[-1][1] += 1
                continue
            y = need[r][i]
            if y in sup or res.get(y) is True:
                top[1] += 1
            elif y not in need or res.get(y) in ("g", False):
                # r thất bại, và mọi món bên dưới trên stack cũng thất bại
                while stack:
                    res[stack.pop()[0]] = False
            else:
                res[y] = "g"; stack.append([y, 0])
    return [r for r in recipes if res[r] is True]

def bt5(recipes, ingredients, supplies):
    if recipes[0] not in set(kahn(recipes, ingredients, supplies)):
        return -1
    need = dict(zip(recipes, ingredients)); sup = set(supplies)
    seen = {recipes[0]}; q = deque([recipes[0]]); base = set()
    while q:
        r = q.popleft()
        for x in need[r]:
            if x in sup: base.add(x)
            elif x not in seen: seen.add(x); q.append(x)
    return len(base)

assert bt3(*examples[2][:3]) == []
assert bt3(["a","b","c"], [["b"],["a"],["x"]], ["x"]) == ["a","b"]
assert bt5(["burger","bread","sandwich"], [["sandwich","meat","bread"],["yeast","flour"],["bread","meat"]], ["yeast","flour","meat","corn"]) == 3
assert bt5(["a","b"], [["b"],["a"]], ["x"]) == -1
assert bt5(["a","b"], [["b","x"],["y"]], ["x","y","z"]) == 2

rng = random.Random(19)
for _ in range(2000):
    r, ing, sup = gen_case(rng)
    ref = brute_fixed_point(r, ing, sup)
    assert bt4(r, ing, sup) == [x for x in r if x in set(ref)]
    assert bt3(r, ing, sup) == sorted(set(r) - set(ref))
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Lưu ý ở `bt4`: khi một nguyên liệu thất bại, **mọi** món đang nằm trên ngăn xếp đều thất bại theo (vì mỗi món trên ngăn xếp cần món ngay trên nó — chuỗi AND). Đây là điểm khác so với DFS tìm đường đi thông thường.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Kahn đa nguồn cho đồ thị phụ thuộc "cần TẤT CẢ"
g = defaultdict(list); indeg = {}
for node, reqs in zip(nodes, requirements):
    indeg[node] = len(reqs)
    for x in reqs: g[x].append(node)
q = deque(sources)              # supplies
ans = []
while q:
    x = q.popleft()
    for v in g.get(x, ()):
        indeg[v] -= 1
        if indeg[v] == 0:
            ans.append(v); q.append(v)
```

**Checklist trước khi submit:**
- [ ] Cạnh đúng chiều: `nguyên liệu → món` (không phải ngược lại).
- [ ] `indeg[món] = len(ingredients[i])` — đề đảm bảo không trùng nguyên liệu nên đếm `len` là đúng.
- [ ] Hàng đợi khởi tạo bằng **supplies** (hoặc đếm bậc bỏ qua supplies — nhất quán một kiểu).
- [ ] Dùng `g.get(x, ())` hoặc `defaultdict` để không `KeyError` với supply không ai cần.
- [ ] Không sửa list input (`list(supplies)`), không dùng `list.pop(0)`.
- [ ] Nếu dùng DFS: có trạng thái xám, gặp xám trả `False`, dùng `is True` chứ không `== True`.
- [ ] Đã thử: chu trình, nguyên liệu ma, chuỗi liệt kê ngược.